In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [4]:
X_train,X_val,y_train,y_val = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [5]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_val = ss.transform(X_val)

In [36]:
X_train_tensors = torch.from_numpy(X_train.astype(np.float32))
X_val_tensors = torch.from_numpy(X_val.astype(np.float32))
y_train_tensors = torch.from_numpy(y_train.astype(np.float32))
y_val_tensors = torch.from_numpy(y_val.astype(np.float32))

# Model Building

In [16]:
import torch.nn as nn

class SimpleNN(nn.Module):

  def __init__(self, input_features):
    super().__init__()
    self.linear1 = nn.Linear(input_features,10)
    self.linear2 = nn.Linear(10,1)
    self.relu = nn.ReLU()
    self.sigmoid = nn.Sigmoid()

  def forward(self,features):
    x = self.linear1(features)
    x = self.relu(x)
    x = self.linear2(x)
    x = self.sigmoid(x)

    return x

In [37]:
y_train_tensors = y_train_tensors.unsqueeze(1)

In [41]:
model = SimpleNN(X_train.shape[1])

lr = 0.1
epochs = 25

loss_function = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(),lr=lr)

for _ in range(epochs):
  y_pred = model(X_train_tensors)
  loss = loss_function(y_pred,y_train_tensors)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  print(f"for epoch{_} the loss is {loss.item()}")

for epoch0 the loss is 0.7287701368331909
for epoch1 the loss is 0.6952660083770752
for epoch2 the loss is 0.6669782400131226
for epoch3 the loss is 0.6424089074134827
for epoch4 the loss is 0.6206015348434448
for epoch5 the loss is 0.6005478501319885
for epoch6 the loss is 0.5813000798225403
for epoch7 the loss is 0.5626082420349121
for epoch8 the loss is 0.5445143580436707
for epoch9 the loss is 0.526782214641571
for epoch10 the loss is 0.5094118714332581
for epoch11 the loss is 0.4923456311225891
for epoch12 the loss is 0.4755229651927948
for epoch13 the loss is 0.4589528441429138
for epoch14 the loss is 0.44267281889915466
for epoch15 the loss is 0.4266868531703949
for epoch16 the loss is 0.41098320484161377
for epoch17 the loss is 0.3956241011619568
for epoch18 the loss is 0.38063934445381165
for epoch19 the loss is 0.36611419916152954
for epoch20 the loss is 0.35208991169929504
for epoch21 the loss is 0.33856895565986633
for epoch22 the loss is 0.3256159722805023
for epoch23 the 

In [39]:
model.eval()

SimpleNN(
  (linear1): Linear(in_features=30, out_features=10, bias=True)
  (linear2): Linear(in_features=10, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)

In [40]:
with torch.no_grad():
  y_pred = model(X_val_tensors)
  pred = (y_pred>0.5).float()
  acc = (pred==y_val).float().mean()

print(acc)

tensor(0.5554)
